# Step 2 — Week 2: Function Calling, Schema Validation & Fallback Prompts

## Goals
- Implement function calling with at least 3 tools
- Build schema-constrained outputs using JSON validation
- Add fallback prompts for malformed outputs

## Prerequisites
- Ollama installed and running: `ollama serve`
- Model pulled: `ollama pull llama3.2`
- Install dependencies: `pip install jsonschema requests`

## Setup

In [ ]:
import requests
import json
import re
from typing import Any, Dict, List, Optional, Callable
from jsonschema import validate, ValidationError, Draft7Validator
from datetime import datetime
import traceback

OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.2"

def ollama_generate(
    prompt: str,
    system: str = "You are a helpful assistant.",
    temperature: float = 0.7,
    top_p: float = 0.9,
    model: str = MODEL,
    max_retries: int = 3
) -> str:
    """Generate text using Ollama API with retry logic."""
    for attempt in range(max_retries):
        try:
            body = {
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": temperature,
                    "top_p": top_p,
                },
            }
            if system:
                body["system"] = system

            response = requests.post(
                f"{OLLAMA_URL}/api/generate",
                json=body,
                timeout=120,
            )
            response.raise_for_status()
            return response.json()["response"].strip()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"Retry {attempt + 1}/{max_retries} - Error: {e}")
            
    return ""

# Smoke test
print("Testing Ollama connection...")
print(ollama_generate("Say: ready", temperature=1.0))

---
## Part 1 — Define Tool Functions (At least 3 tools)

In [ ]:
# Tool 1: Calculator
def calculator(operation: str, a: float, b: float) -> Dict[str, Any]:
    """
    Perform basic mathematical operations.
    
    Args:
        operation: One of 'add', 'subtract', 'multiply', 'divide'
        a: First number
        b: Second number
    
    Returns:
        Dictionary with result and operation details
    """
    try:
        if operation == "add":
            result = a + b
        elif operation == "subtract":
            result = a - b
        elif operation == "multiply":
            result = a * b
        elif operation == "divide":
            if b == 0:
                return {"error": "Division by zero", "operation": operation}
            result = a / b
        else:
            return {"error": f"Unknown operation: {operation}"}
        
        return {
            "operation": operation,
            "a": a,
            "b": b,
            "result": result,
            "success": True
        }
    except Exception as e:
        return {"error": str(e), "operation": operation, "success": False}


# Tool 2: Weather Lookup (simulated)
def get_weather(city: str, unit: str = "celsius") -> Dict[str, Any]:
    """
    Get weather information for a city.
    
    Args:
        city: City name
        unit: Temperature unit ('celsius' or 'fahrenheit')
    
    Returns:
        Dictionary with weather information
    """
    # Simulated weather data
    weather_db = {
        "new york": {"temp": 15, "condition": "cloudy", "humidity": 65},
        "london": {"temp": 12, "condition": "rainy", "humidity": 80},
        "paris": {"temp": 18, "condition": "sunny", "humidity": 55},
        "tokyo": {"temp": 22, "condition": "partly cloudy", "humidity": 70},
    }
    
    city_lower = city.lower()
    if city_lower not in weather_db:
        return {
            "error": f"City '{city}' not found in database",
            "success": False
        }
    
    data = weather_db[city_lower]
    temp = data["temp"]
    
    # Convert to Fahrenheit if requested
    if unit.lower() == "fahrenheit":
        temp = (temp * 9/5) + 32
    
    return {
        "city": city,
        "temperature": temp,
        "unit": unit,
        "condition": data["condition"],
        "humidity": data["humidity"],
        "success": True
    }


# Tool 3: Database Query (simulated)
def query_database(table: str, filter_field: str = "", filter_value: str = "") -> Dict[str, Any]:
    """
    Query a simulated database.
    
    Args:
        table: Table name ('users', 'products', 'orders')
        filter_field: Field to filter by
        filter_value: Value to filter for
    
    Returns:
        Query results
    """
    # Simulated database
    db = {
        "users": [
            {"id": 1, "name": "Alice", "email": "alice@example.com", "status": "active"},
            {"id": 2, "name": "Bob", "email": "bob@example.com", "status": "inactive"},
            {"id": 3, "name": "Charlie", "email": "charlie@example.com", "status": "active"},
        ],
        "products": [
            {"id": 1, "name": "Laptop", "price": 999.99, "stock": 5},
            {"id": 2, "name": "Mouse", "price": 29.99, "stock": 50},
            {"id": 3, "name": "Keyboard", "price": 79.99, "stock": 20},
        ],
        "orders": [
            {"id": 1, "user_id": 1, "product_id": 1, "quantity": 1, "status": "delivered"},
            {"id": 2, "user_id": 2, "product_id": 2, "quantity": 2, "status": "pending"},
        ]
    }
    
    if table not in db:
        return {"error": f"Table '{table}' not found", "success": False}
    
    rows = db[table]
    
    # Apply filter if provided
    if filter_field and filter_value:
        rows = [r for r in rows if str(r.get(filter_field)) == str(filter_value)]
    
    return {
        "table": table,
        "rows": rows,
        "count": len(rows),
        "filter": {"field": filter_field, "value": filter_value} if filter_field else None,
        "success": True
    }


# Store tools in a registry
TOOLS_REGISTRY = {
    "calculator": calculator,
    "get_weather": get_weather,
    "query_database": query_database,
}

print("✓ Tool functions defined successfully")
print("Available tools:", list(TOOLS_REGISTRY.keys()))

---
## Part 2 — Create Function Calling Schema

Define JSON schemas for each tool that describe:
- Function name
- Human-readable description
- Parameters (with types and descriptions)

This follows the OpenAI function calling format.

In [ ]:
FUNCTION_SCHEMAS = {
    "calculator": {
        "name": "calculator",
        "description": "Perform basic mathematical operations (add, subtract, multiply, divide)",
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "description": "Mathematical operation to perform",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "a": {
                    "type": "number",
                    "description": "First number"
                },
                "b": {
                    "type": "number",
                    "description": "Second number"
                }
            },
            "required": ["operation", "a", "b"]
        }
    },
    "get_weather": {
        "name": "get_weather",
        "description": "Get current weather information for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Name of the city"
                },
                "unit": {
                    "type": "string",
                    "description": "Temperature unit",
                    "enum": ["celsius", "fahrenheit"],
                    "default": "celsius"
                }
            },
            "required": ["city"]
        }
    },
    "query_database": {
        "name": "query_database",
        "description": "Query the database for information",
        "parameters": {
            "type": "object",
            "properties": {
                "table": {
                    "type": "string",
                    "description": "Name of the table to query",
                    "enum": ["users", "products", "orders"]
                },
                "filter_field": {
                    "type": "string",
                    "description": "Field to filter by (optional)"
                },
                "filter_value": {
                    "type": "string",
                    "description": "Value to filter for (optional)"
                }
            },
            "required": ["table"]
        }
    }
}

# Display schemas in a readable format
print("Function Calling Schemas:")
print("=" * 80)
for tool_name, schema in FUNCTION_SCHEMAS.items():
    print(f"\n📌 {schema['name']}")
    print(f"   Description: {schema['description']}")
    print(f"   Parameters: {json.dumps(schema['parameters'], indent=6)}")
    print()

---
## Part 3 — Implement Function Calling with LLM

Since Ollama doesn't have native function calling like OpenAI, we'll instruct the LLM to
output JSON with the function to call and its parameters. The LLM will choose which tool
to use based on the user's query.

In [ ]:
def build_function_prompt(user_query: str) -> str:
    """Build a prompt that instructs the LLM to choose and call a function."""
    
    schemas_text = json.dumps(FUNCTION_SCHEMAS, indent=2)
    
    prompt = f"""You are an AI assistant with access to the following tools. Call the appropriate tool to help the user.

Available tools:
{schemas_text}

User request: {user_query}

Respond with a JSON object in this exact format:
{{
    "function": "function_name",
    "parameters": {{
        "param1": value1,
        "param2": value2
    }},
    "reasoning": "Brief explanation of why you chose this tool"
}}

Only return the JSON object, no other text."""
    
    return prompt


def parse_function_call(response_text: str) -> Optional[Dict[str, Any]]:
    """
    Extract function call JSON from LLM response.
    Returns None if parsing fails.
    """
    try:
        # Try to find JSON in the response
        json_match = re.search(r'\{[\s\S]*\}', response_text)
        if json_match:
            json_str = json_match.group(0)
            return json.loads(json_str)
    except (json.JSONDecodeError, AttributeError) as e:
        print(f"Failed to parse function call: {e}")
    
    return None


def call_function(function_name: str, parameters: Dict[str, Any]) -> Dict[str, Any]:
    """Execute a function call with given parameters."""
    
    if function_name not in TOOLS_REGISTRY:
        return {
            "error": f"Unknown function: {function_name}",
            "success": False
        }
    
    try:
        func = TOOLS_REGISTRY[function_name]
        result = func(**parameters)
        return result
    except TypeError as e:
        return {
            "error": f"Invalid parameters for {function_name}: {e}",
            "success": False
        }
    except Exception as e:
        return {
            "error": f"Error executing {function_name}: {e}",
            "success": False
        }


# Test the function calling system
test_queries = [
    "What is 15 plus 27?",
    "Tell me the weather in Paris in Fahrenheit",
    "Show me all active users in the database"
]

print("Testing Function Calling System")
print("=" * 80)

for query in test_queries:
    print(f"\n📝 User: {query}")
    
    # Build prompt
    prompt = build_function_prompt(query)
    
    # Get LLM response
    llm_response = ollama_generate(prompt, temperature=0.3)
    print(f"🤖 LLM Response:\n{llm_response}\n")
    
    # Parse function call
    func_call = parse_function_call(llm_response)
    
    if func_call:
        print(f"✓ Parsed function call: {func_call.get('function')}")
        print(f"  Reasoning: {func_call.get('reasoning')}")
        
        # Execute function
        result = call_function(func_call['function'], func_call.get('parameters', {}))
        print(f"📊 Result:\n{json.dumps(result, indent=2)}\n")
    else:
        print("✗ Failed to parse function call\n")

---
## Part 4 — Build Schema-Constrained JSON Output

Define strict JSON schemas for expected LLM outputs. This ensures outputs follow
a specific structure before being processed by your application.

In [ ]:
# Output schema for a movie summary
MOVIE_SUMMARY_SCHEMA = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "minLength": 1,
            "description": "Movie title"
        },
        "year": {
            "type": "integer",
            "minimum": 1800,
            "maximum": 2100,
            "description": "Release year"
        },
        "genre": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1,
            "maxItems": 5,
            "description": "Movie genres"
        },
        "rating": {
            "type": "number",
            "minimum": 0,
            "maximum": 10,
            "description": "IMDb rating"
        },
        "summary": {
            "type": "string",
            "minLength": 50,
            "maxLength": 500,
            "description": "Brief plot summary"
        },
        "cast": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1,
            "maxItems": 5,
            "description": "Main cast members"
        }
    },
    "required": ["title", "year", "genre", "rating", "summary", "cast"],
    "additionalProperties": False
}

# Output schema for an analysis
ANALYSIS_SCHEMA = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "type": "object",
    "properties": {
        "question": {
            "type": "string",
            "description": "The original question"
        },
        "answer": {
            "type": "string",
            "minLength": 10,
            "description": "The analysis answer"
        },
        "confidence": {
            "type": "number",
            "minimum": 0,
            "maximum": 1,
            "description": "Confidence level (0-1)"
        },
        "sources": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Information sources"
        },
        "key_points": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1,
            "maxItems": 5,
            "description": "Key points from the analysis"
        }
    },
    "required": ["question", "answer", "confidence", "key_points"],
    "additionalProperties": False
}

print("✓ Output schemas defined:")
print("  - MOVIE_SUMMARY_SCHEMA")
print("  - ANALYSIS_SCHEMA")

---
## Part 5 — Validate Output Against Schema

Use jsonschema to validate LLM outputs match the expected structure.
This ensures only well-formed data enters your application.

In [ ]:
def validate_json_output(data: Dict[str, Any], schema: Dict[str, Any]) -> tuple[bool, Optional[str]]:
    """
    Validate JSON output against a schema.
    
    Returns:
        Tuple of (is_valid, error_message)
    """
    try:
        validate(instance=data, schema=schema)
        return (True, None)
    except ValidationError as e:
        error_msg = f"Validation error at '{'.'.join(str(p) for p in e.absolute_path)}': {e.message}"
        return (False, error_msg)
    except Exception as e:
        return (False, f"Unexpected validation error: {str(e)}")


def extract_json_from_response(response_text: str) -> Optional[Dict[str, Any]]:
    """Extract JSON object from LLM response text."""
    try:
        json_match = re.search(r'\{[\s\S]*\}', response_text)
        if json_match:
            return json.loads(json_match.group(0))
    except (json.JSONDecodeError, AttributeError):
        pass
    return None


# Test JSON validation with valid and invalid examples
print("Testing JSON Schema Validation")
print("=" * 80)

# Valid movie summary
valid_movie = {
    "title": "The Matrix",
    "year": 1999,
    "genre": ["Sci-Fi", "Action"],
    "rating": 8.7,
    "summary": "A computer hacker learns about the true nature of his reality and his role in the war against its controllers.",
    "cast": ["Keanu Reeves", "Laurence Fishburne", "Carrie-Anne Moss"]
}

print("\n✅ Valid Movie Summary:")
is_valid, error = validate_json_output(valid_movie, MOVIE_SUMMARY_SCHEMA)
print(f"Valid: {is_valid}")
if error:
    print(f"Error: {error}")

# Invalid movie summary (missing required field)
invalid_movie_1 = {
    "title": "The Matrix",
    "year": 1999,
    "genre": ["Sci-Fi", "Action"],
    "rating": 8.7,
    # Missing "summary" field
    "cast": ["Keanu Reeves", "Laurence Fishburne"]
}

print("\n❌ Invalid Movie (missing summary):")
is_valid, error = validate_json_output(invalid_movie_1, MOVIE_SUMMARY_SCHEMA)
print(f"Valid: {is_valid}")
if error:
    print(f"Error: {error}")

# Invalid movie summary (rating out of range)
invalid_movie_2 = {
    "title": "The Matrix",
    "year": 1999,
    "genre": ["Sci-Fi", "Action"],
    "rating": 15.0,  # Out of range
    "summary": "A computer hacker learns about the true nature of his reality.",
    "cast": ["Keanu Reeves"]
}

print("\n❌ Invalid Movie (rating > 10):")
is_valid, error = validate_json_output(invalid_movie_2, MOVIE_SUMMARY_SCHEMA)
print(f"Valid: {is_valid}")
if error:
    print(f"Error: {error}")

# Valid analysis
valid_analysis = {
    "question": "What are the benefits of machine learning?",
    "answer": "Machine learning enables systems to learn from data and improve performance without explicit programming.",
    "confidence": 0.92,
    "sources": ["ML textbooks", "academic papers"],
    "key_points": ["Automation", "Pattern recognition", "Scalability"]
}

print("\n✅ Valid Analysis:")
is_valid, error = validate_json_output(valid_analysis, ANALYSIS_SCHEMA)
print(f"Valid: {is_valid}")
if error:
    print(f"Error: {error}")

---
## Part 6 — Implement Fallback Prompts for Malformed Outputs

When the LLM output doesn't match the schema, use fallback prompts to request
properly formatted data. Include retry logic and error reporting.

In [ ]:
def get_fallback_prompt(original_prompt: str, error_message: str, schema: Dict[str, Any], attempt: int) -> str:
    """Generate a fallback prompt when output validation fails."""
    
    schema_str = json.dumps(schema, indent=2)
    
    if attempt == 1:
        fallback = f"""Your previous response didn't match the required format.

Error: {error_message}

Required JSON Schema:
{schema_str}

Please respond with ONLY valid JSON that matches this schema. No additional text."""
    
    elif attempt == 2:
        fallback = f"""The last two responses didn't match the required format.

Error: {error_message}

Required JSON Schema:
{schema_str}

CRITICAL: Respond with ONLY the JSON object, nothing else. Make sure:
1. All required fields are present
2. All field types match the schema
3. No extra fields are included
4. All values are valid for their types"""
    
    else:  # attempt >= 3
        fallback = f"""Third attempt: Your responses keep failing validation.

Error: {error_message}

Expected JSON Schema:
{schema_str}

RESPOND WITH ONLY THIS JSON FORMAT. DO NOT ADD EXPLANATION, MARKDOWN BLOCKS, OR ANY OTHER TEXT.

Start with {{ and end with }}."""
    
    return fallback


def validate_with_retry(
    prompt: str,
    schema: Dict[str, Any],
    max_retries: int = 3,
    temperature: float = 0.3
) -> tuple[Optional[Dict[str, Any]], int, List[str]]:
    """
    Try to get valid JSON from LLM with automatic retries.
    
    Returns:
        Tuple of (valid_data, attempts_used, error_messages)
    """
    errors = []
    
    for attempt in range(max_retries):
        if attempt == 0:
            current_prompt = prompt
            print(f"\n🔄 Attempt {attempt + 1}/{max_retries}")
        else:
            # Use fallback prompt
            error_msg = errors[-1] if errors else "Unknown error"
            current_prompt = get_fallback_prompt(prompt, error_msg, schema, attempt)
            print(f"\n🔄 Attempt {attempt + 1}/{max_retries} (Fallback)")
        
        # Get LLM response
        response = ollama_generate(current_prompt, temperature=temperature)
        print(f"Response: {response[:100]}..." if len(response) > 100 else f"Response: {response}")
        
        # Try to extract JSON
        data = extract_json_from_response(response)
        
        if data is None:
            error = "Could not extract JSON from response"
            errors.append(error)
            print(f"❌ {error}")
            continue
        
        # Validate against schema
        is_valid, error = validate_json_output(data, schema)
        
        if is_valid:
            print(f"✅ Valid output received!")
            return (data, attempt + 1, errors)
        else:
            errors.append(error)
            print(f"❌ {error}")
    
    # All retries exhausted
    print(f"\n❌ Failed after {max_retries} attempts")
    return (None, max_retries, errors)


# Test the fallback system
print("\n" + "=" * 80)
print("Testing Fallback Prompt System")
print("=" * 80)

# Create a base prompt for movie summary
movie_prompt = f"""Provide a JSON summary of the movie 'Inception' (2010).

Required JSON Schema:
{json.dumps(MOVIE_SUMMARY_SCHEMA, indent=2)}

Respond with ONLY the JSON object."""

print(f"Prompting for movie summary...")
valid_movie, attempts, errors = validate_with_retry(
    movie_prompt,
    MOVIE_SUMMARY_SCHEMA,
    max_retries=3,
    temperature=0.2
)

if valid_movie:
    print(f"\n✅ SUCCESS! Got valid movie data in {attempts} attempt(s)")
    print(json.dumps(valid_movie, indent=2))
else:
    print(f"\n❌ FAILED after {attempts} attempts")
    print("\nErrors encountered:")
    for i, error in enumerate(errors, 1):
        print(f"  {i}. {error}")

---
## Part 7 — Test Function Calling Pipeline

End-to-end testing combining:
- Function calling (user query → function selection)
- Schema validation (ensuring proper JSON structure)
- Fallback handling (retries with better prompts)

In [ ]:
class FunctionCallingPipeline:
    """End-to-end pipeline for function calling with validation and fallback."""
    
    def __init__(self, schemas: Dict[str, Dict], max_retries: int = 3):
        self.schemas = schemas
        self.max_retries = max_retries
        self.call_history = []
    
    def execute_user_query(self, user_query: str) -> Dict[str, Any]:
        """
        Execute a complete pipeline for a user query:
        1. Determine which function to call
        2. Execute the function with validated parameters
        3. Return formatted result
        """
        timestamp = datetime.now().isoformat()
        
        print(f"\n{'='*80}")
        print(f"📞 User Query: {user_query}")
        print(f"{'='*80}")
        
        # Step 1: Get function call from LLM
        prompt = build_function_prompt(user_query)
        llm_response = ollama_generate(prompt, temperature=0.3)
        
        func_call = parse_function_call(llm_response)
        if not func_call:
            return {
                "success": False,
                "error": "Failed to parse function call",
                "timestamp": timestamp
            }
        
        function_name = func_call.get('function')
        parameters = func_call.get('parameters', {})
        
        print(f"✓ Function selected: {function_name}")
        print(f"  Reasoning: {func_call.get('reasoning', 'N/A')}")
        print(f"  Parameters: {parameters}")
        
        # Step 2: Execute the function
        result = call_function(function_name, parameters)
        
        # Step 3: Log to history
        self.call_history.append({
            "timestamp": timestamp,
            "user_query": user_query,
            "function": function_name,
            "parameters": parameters,
            "result": result
        })
        
        return {
            "success": result.get('success', False),
            "function": function_name,
            "parameters": parameters,
            "result": result,
            "timestamp": timestamp
        }
    
    def generate_analysis(self, question: str) -> Dict[str, Any]:
        """Generate an analysis with validated JSON output."""
        
        print(f"\n{'='*80}")
        print(f"📊 Analysis Request: {question}")
        print(f"{'='*80}")
        
        analysis_prompt = f"""Analyze this question and provide insights:
Question: {question}

Respond with JSON in this exact format:
{{
    "question": "{{the question}}",
    "answer": "{{your detailed answer}}",
    "confidence": {{0.0-1.0}},
    "sources": ["{{source1}}", "{{source2}}"],
    "key_points": ["{{point1}}", "{{point2}}", "{{point3}}"]
}}

Only respond with the JSON object."""
        
        valid_analysis, attempts, errors = validate_with_retry(
            analysis_prompt,
            ANALYSIS_SCHEMA,
            max_retries=3,
            temperature=0.2
        )
        
        if valid_analysis:
            print(f"\n✅ Analysis successful!")
            return {
                "success": True,
                "data": valid_analysis,
                "attempts": attempts
            }
        else:
            print(f"\n❌ Analysis failed after {attempts} attempts")
            return {
                "success": False,
                "errors": errors,
                "attempts": attempts
            }
    
    def show_history(self):
        """Display call history."""
        print(f"\n{'='*80}")
        print("📜 Function Call History")
        print(f"{'='*80}")
        
        if not self.call_history:
            print("No function calls made yet.")
            return
        
        for i, call in enumerate(self.call_history, 1):
            print(f"\n{i}. {call['timestamp']}")
            print(f"   Query: {call['user_query']}")
            print(f"   Function: {call['function']}")
            print(f"   Result: {'✓ Success' if call['result'].get('success') else '✗ Failed'}")


# Run end-to-end test
print("\n" + "="*80)
print("COMPREHENSIVE END-TO-END PIPELINE TEST")
print("="*80)

pipeline = FunctionCallingPipeline(FUNCTION_SCHEMAS, max_retries=3)

# Test 1: Function calling with calculation
print("\n--- Test 1: Function Calling (Calculator) ---")
pipeline.execute_user_query("What is 42 times 13?")

# Test 2: Function calling with database query
print("\n--- Test 2: Function Calling (Database) ---")
pipeline.execute_user_query("Show me all products with their prices")

# Test 3: Analysis with schema validation
print("\n--- Test 3: Analysis with Schema Validation ---")
pipeline.generate_analysis("How do neural networks learn patterns in data?")

# Show history
pipeline.show_history()

print("\n" + "="*80)
print("✅ ALL TESTS COMPLETED")
print("="*80)

---
## Key Takeaways

### 🎯 Function Calling
- Define tool functions with clear parameter specifications
- Create JSON schemas following OpenAI function calling format
- Instruct the LLM to output JSON with function name and parameters
- Parse LLM responses to extract and execute the requested functions

### ✅ Schema Validation
- Use jsonschema library to validate JSON against defined schemas
- Define strict schemas with required fields, types, and constraints
- Catch malformed outputs before they enter your application
- Validate against multiple schemas for different response types

### 🔄 Fallback Prompts
- Detect validation failures and track error messages
- Generate progressive fallback prompts that increase clarity
- Implement retry logic with exponential prompting (1st, 2nd, 3rd attempts)
- Set maximum retry limits to prevent infinite loops

### 🏗️ Best Practices
1. **Clear Tool Documentation**: Detailed descriptions in schemas help LLMs choose correctly
2. **Type Safety**: Strict JSON schemas prevent bugs and edge cases
3. **User Feedback**: Track validation failures to improve prompts over time
4. **Temperature Control**: Use lower temperatures (0.2-0.3) for structured outputs
5. **Error Handling**: Always handle cases where the LLM can't generate valid output

### 📚 Further Exploration
- Experiment with different temperature values for function calling
- Add more complex tools with nested parameters
- Implement caching to avoid re-calling the same functions
- Use confidence scores to prioritize which outputs to trust
- Combine function calling with retrieval-augmented generation (RAG)